<a href="https://colab.research.google.com/github/matinapap/Multimedia-Systems/blob/main/Multimedia_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import shutil
import os

# Λίστα με τους φακέλους που θέλουμε να διαγράψουμε
folders_to_delete = ["results"]

for folder in folders_to_delete:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Deleted folder: {folder}")
    else:
        print(f"Folder not found, skipping: {folder}")

Deleted folder: results


### Επιλογή βίντεο:

In [ ]:
from google.colab import files
uploaded = files.upload()  # διάλεξε το video.avi

Saving input_video.avi to input_video (1).avi


### Google Drive:

In [ ]:
# -------------------------------
# Google Drive mount & έλεγχος αρχείων
# -------------------------------
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# -------------------------------
# Import βιβλιοθήκες και τοποθεσία βίντεο
# -------------------------------
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import heapq
from collections import Counter

video_path = '/content/drive/MyDrive/input_video.avi'

In [ ]:
# -------------------------------
# (A) Frame differences
# -------------------------------
"""
1. Ανοίγει το βίντεο
2. Διαβάζει το πρώτο καρέ και το μετατρέπει σε grayscale
3. Υπολογίζει διαφορές μεταξύ διαδοχικών καρέ
4. Αποθηκεύει τις εικόνες διαφορών σε λίστα
"""
def compute_frame_differences(video_path):
    # φορτώνει το βίντεο
    cap = cv2.VideoCapture(video_path)

    # ελέγχει αν το βίντεο ανοίγει
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    ret, prev_frame = cap.read()
    if not ret:
        raise IOError("Cannot read the first frame from the video")

    # μετατρέπει το πρώτο frame σε grayscale
    prev_frame_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    # λίστα που αποθηκεύει όλα τα frame differences
    frame_diffs = []

    while True:
        # διαβάζει το επόμενο frame
        ret, frame = cap.read()
        if not ret:
            break
        # μετατρέπει το τωρινό frame σε grayscale
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # υπολογίζει την απόλυτη διαφορά μεταξύ του τωρινού και του προηγούμενου frame
        diff = cv2.absdiff(frame_gray, prev_frame_gray)

        # αποθηκεύει την διαφορά της εικόνας
        frame_diffs.append(diff)

        # ενημερώνει το previous frame
        prev_frame_gray = frame_gray

    # απελευθερώνει το αντικείμενο video capture
    cap.release()

    return frame_diffs

def visualize_frame_differences(frame_diffs, output_dir, prefix="diff"):
    # εξασφαλίζει οτι υπάρχει φάκελος για το output
    os.makedirs(output_dir, exist_ok=True)

    # επανάληψη για όλα τα διαφορετικά frames
    for i, diff in enumerate(frame_diffs):
        plt.imshow(diff, cmap='gray') # εμφανίζει την difference image σε grayscale
        plt.axis('off') # αφαιρεί τον άξονα X και Y
        plt.title(f'{prefix} {i}')  # γράφει ποια εικόνα διαφορών βλέπεις
        plt.savefig(os.path.join(output_dir, f'{prefix}_{i:04d}.png')) # αποθηκεύει την εικόνα σε PNG στο output directory
        plt.close()

In [ ]:
# -------------------------------
# (B) Huffman coding
# -------------------------------
"""
1. Υπολογίζει την συχνότητα των συμβόλων
2. Δημιουργεί Huffman δέντρο
3. Αποδίδει κωδικούς και επιστρέφει τα στατιστικά
"""
def compute_huffman(frame_diffs, output_dir="huffman_output"):
    # εξασφαλίζει οτι υπάρχει φάκελος για το output
    os.makedirs(output_dir, exist_ok=True)

    # μέτρημα συχνοτήτων συμβόλων
    freq = Counter()
    total_symbols = 0
    for diff in frame_diffs:
        arr = diff.ravel()  # μετατρέπει την εικόνα σε 1D array
        freq.update(arr.tolist()) # μετράει πόσες φορές εμφανίζεται κάθε pixel value
        total_symbols += arr.size # μετράει το συνολικό πλήθος συμβόλων

    # έλεγχος αν υπάρχουν σύμβολα
    if total_symbols == 0:
        print("No symbols found in frame_diffs.")
        return {}, 0, 0

    # δημιουργεί μία ελάχιστη ουρά προτεραιότητας (min-heap) για τα σύμβολα, με βάση τη συχνότητα τους
    heap = []
    uid = 0
    # ουρά προτεραιότητας
    for sym, f in freq.items():
        heap.append((f, uid, sym))
        uid += 1
    heapq.heapify(heap)

    # κατασκευή Huffman δέντρου
    if len(heap) == 1:
        _, _, only_sym = heap[0]
        codes = {int(only_sym): "0"}  # αν υπάρχει μόνο ένα σύμβολο δίνει κωδικό 0
    else:
        while len(heap) > 1: # διαφορετικά φτιάχνει Huffman Tree
            f1, id1, node1 = heapq.heappop(heap)
            f2, id2, node2 = heapq.heappop(heap)
            combined = (f1 + f2, uid, (node1, node2))
            uid += 1
            heapq.heappush(heap, combined)
        _, _, root = heap[0]


        # απόδοση Huffman κωδικών
        # διασχίζει το δέντρο και δίνει 0 όταν πάει αριστερά και 1 όταν πάει δεξιά
        codes = {}
        def assign_codes(node, prefix):
            if isinstance(node, int) or isinstance(node, np.integer):
                codes[int(node)] = prefix if prefix != "" else "0"
                return
            left, right = node
            assign_codes(left, prefix + "0")
            assign_codes(right, prefix + "1")
        assign_codes(root, "")

    compressed_bits = sum(freq[sym] * len(codes[int(sym)]) for sym in freq)  # πόσα bits χρειάζονται με το Huffman
    original_bits = total_symbols * 8 # πόσα bits θα χρειάζονταν αν κωδικοποιούσαμε όλα τα pixel σε 8-bit grayscale.

    # αποθηκεύεται σε .txt για κάθε pixel value: το σύμβολο (π.χ. 0, 128, 255), την συχνότητα εμφάνισης, τον Huffman κωδικό του και το μήκος του κωδικού
    codebook_path = os.path.join(output_dir, "huffman_codebook.txt")
    with open(codebook_path, "w") as fh:
        fh.write("symbol\tfrequency\tcode\tcode_length\n")
        for sym, f in freq.most_common():
            fh.write(f"{int(sym)}\t{f}\t{codes[int(sym)]}\t{len(codes[int(sym)])}\n")

    # εμφανίζονται τα στατιστικά: μέγεθος αρχικού αρχείου (σε bits), μέγεθος μετά τη συμπίεση, αναλογία συμπίεσης (compression ration) και μέσο αριθμό bits ανά σύμβολο
    print(f"Ο Huffman αποθηκεύτηκε στο: {codebook_path}")
    print(f"Αρχικά bits: {original_bits}")
    print(f"Μέγεθος Συμπιεσμένης Δυαδικής ροής: {compressed_bits}")
    if compressed_bits > 0:
        print(f"Λόγος Συμπίεσης: {original_bits / compressed_bits:.4f}")
        print(f"Μέσο αριθό ανά bits/symbol: {compressed_bits / total_symbols:.4f}")

    return codes, original_bits, compressed_bits

In [ ]:
# -------------------------------
# (C) Motion Compensation
# -------------------------------
"""
1. Διαβάζει τα καρέ του βίντεο
2. Κάνει Block Matching (εύρεση μετακίνησης μπλοκ)
3. Υπολογίζει τις διαφορές και τις αποθηκεύει
"""
def block_matching(prev_frame, curr_frame, block_size=16, search_range=8):
    # παίρνουμε το ύψος και το πλάτος του frame
    h, w = curr_frame.shape
    # δημιουργούμε ένα frame πρόβλεψης αρχικά μηνεδικό
    pred_frame = np.zeros_like(curr_frame)

    for y in range(0, h, block_size):
        for x in range(0, w, block_size):
            # παρακάμπτει μπλοκ που ξεπερνούν τα όρια του frame
            if y + block_size > h or x + block_size > w:
                continue

            block = curr_frame[y:y+block_size, x:x+block_size] # παίρουμε το τρέχον μπλόκ
            best_sad = float("inf")
            best_match = (y, x)

            # ψάχνει στο prev_frame γύρω από την ίδια θέση μέσα σε ένα search range για να βρει το καλύτερο μπλοκ
            for dy in range(-search_range, search_range+1):
                for dx in range(-search_range, search_range+1):
                    ref_y = y + dy
                    ref_x = x + dx
                    if (ref_y < 0 or ref_x < 0 or
                        ref_y + block_size > h or
                        ref_x + block_size > w):
                        continue
                    ref_block = prev_frame[ref_y:ref_y+block_size,
                                            ref_x:ref_x+block_size]
                    # υπολογίζουμε το SAD, μετράμε την διαφορά των pixels μεταξύ των block και το block με την μικρότερη SAD είναι το best match
                    sad = np.sum(np.abs(block.astype(int) - ref_block.astype(int)))
                    if sad < best_sad:
                        best_sad = sad
                        best_match = (ref_y, ref_x)

            ref_y, ref_x = best_match
            # τοποθετεί το καλύτερο μπλοκ στο pred_frame
            pred_frame[y:y+block_size, x:x+block_size] = prev_frame[ref_y:ref_y+block_size,
                                                                    ref_x:ref_x+block_size]
    return pred_frame

# Διαβάζει ένα βίντεο καρέ-καρέ, εκτελεί block matching για κάθε ζεύγος frames και υπολογίζει τη διαφορά αναμεσα στο τρέχον frame και το προβλεπόμενο frame
def compute_motion_compensation(video_path, block_size=32, search_range=8):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    ret, prev_frame = cap.read()
    if not ret:
        raise IOError("Cannot read the first frame from the video")

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    mc_diffs = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        pred = block_matching(prev_gray, curr_gray,
                              block_size=block_size,
                              search_range=search_range)
        diff = cv2.absdiff(curr_gray, pred)
        mc_diffs.append(diff)

        prev_gray = curr_gray

    cap.release()
    return mc_diffs

In [ ]:
output_dir = "results"

# (A) Frame differences
print("---------------------------- (A) Frame differences ----------------------------")
frame_diffs = compute_frame_differences(video_path)
print(f"Computed {len(frame_diffs)} frame differences.")

diffs_dir = os.path.join(output_dir, "frame_diffs_A")
visualize_frame_differences(frame_diffs, diffs_dir, prefix="diff")

---------------------------- (A) Frame differences ----------------------------
Computed 113 frame differences.


In [ ]:
# (B) Huffman coding
print("---------------------------- (B) Huffman coding ----------------------------")
huffman_dir = os.path.join(output_dir, "huffman_B")
compute_huffman(frame_diffs, huffman_dir)

---------------------------- (B) Huffman coding ----------------------------
Ο Huffman αποθηκεύτηκε στο: results/huffman_B/huffman_codebook.txt
Αρχικά bits: 1874534400
Μέγεθος Συμπιεσμένης Δυαδικής ροής: 690429588
Λόγος Συμπίεσης: 2.7150
Μέσο αριθό ανά bits/symbol: 2.9466


({3: '0000',
  9: '000100',
  15: '0001010',
  45: '0001011000',
  46: '0001011001',
  32: '000101101',
  22: '00010111',
  5: '00011',
  31: '001000000',
  64: '00100000100',
  145: '0010000010100000',
  156: '00100000101000010',
  157: '00100000101000011',
  137: '001000001010001',
  123: '00100000101001',
  125: '00100000101010',
  126: '00100000101011',
  78: '001000001011',
  60: '00100000110',
  59: '00100000111',
  102: '0010000100000',
  108: '0010000100001',
  77: '001000010001',
  61: '00100001001',
  44: '0010000101',
  62: '00100001100',
  103: '0010000110100',
  107: '0010000110101',
  76: '001000011011',
  63: '00100001110',
  73: '001000011110',
  165: '001000011111000000',
  174: '0010000111110000010',
  175: '0010000111110000011',
  167: '001000011111000010',
  173: '0010000111110000110',
  176: '0010000111110000111',
  144: '0010000111110001',
  155: '00100001111100100',
  152: '00100001111100101',
  153: '00100001111100110',
  154: '00100001111100111',
  127: '001000

In [ ]:
# (C) Motion-compensated differences
print("---------------------------- (C) Motion-compensated differences ----------------------------")
mc_dir = os.path.join(output_dir, "mc_diffs_C")

# Υπολογισμός motion-compensated differences
mc_diffs = compute_motion_compensation(video_path,
                                       block_size=32,
                                       search_range=8)
print(f"Computed {len(mc_diffs)} MC frame differences.")

# Οπτικοποίηση
visualize_frame_differences(mc_diffs, mc_dir, prefix="mc_diff")

# Huffman coding για τις MC διαφορές
compute_huffman(mc_diffs, os.path.join(output_dir, "huffman_C"))

---------------------------- (C) Motion-compensated differences ----------------------------
Computed 113 MC frame differences.
Ο Huffman αποθηκεύτηκε στο: results/huffman_C/huffman_codebook.txt
Αρχικά bits: 1874534400
Μέγεθος Συμπιεσμένης Δυαδικής ροής: 497511204
Λόγος Συμπίεσης: 3.7678
Μέσο αριθό ανά bits/symbol: 2.1232


({1: '00',
  7: '010000',
  15: '01000100',
  22: '010001010',
  31: '0100010110',
  32: '0100010111',
  9: '0100011',
  47: '01001000000',
  46: '01001000001',
  64: '010010000100',
  82: '0100100001010',
  85: '0100100001011',
  60: '010010000110',
  62: '010010000111',
  79: '0100100010000',
  121: '010010001000100',
  180: '0100100010001010000000',
  182: '0100100010001010000001',
  176: '010010001000101000001',
  166: '01001000100010100001',
  167: '01001000100010100010',
  170: '01001000100010100011',
  149: '010010001000101001',
  140: '01001000100010101',
  134: '0100100010001011',
  107: '01001000100011',
  63: '010010001001',
  61: '010010001010',
  59: '010010001011',
  30: '0100100011',
  21: '010010010',
  20: '010010011',
  14: '01001010',
  44: '01001011000',
  103: '01001011001000',
  102: '01001011001001',
  130: '0100101100101000',
  133: '0100101100101001',
  117: '010010110010101',
  138: '01001011001011000',
  159: '0100101100101100100',
  165: '0100101100101100101